# Numerosity-PRF R² across scanners

Loads the per-(subject, session) R² maps produced by `balgrist/encoding_model/fit_mapper.py` and compares their distributions across the four scanner sessions.

Sessions: `ses-balgrist3t`, `ses-balgrist7t`, `ses-ibt7t`, `ses-sns3t`.

Voxels outside the fitting mask have R² = 0 exactly; we drop them with `r2 != 0`, which is equivalent to a brain-mask filter without needing the masks locally.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from nilearn import image

BIDS_FOLDER = Path('~/data/ds-balgrist').expanduser()
# BIDS_FOLDER = Path('/shares/zne.uzh/gdehol/ds-balgrist')   # on cluster
DERIV = 'encoding_model'
SESSIONS = ('balgrist3t', 'balgrist7t', 'ibt7t', 'sns3t')
SES_ORDER = ['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t']

In [ ]:
def _r2_path(subject, session, bids_folder=BIDS_FOLDER, deriv=DERIV):
    return (bids_folder / 'derivatives' / deriv
            / f'sub-{subject}' / f'ses-{session}' / 'func'
            / f'sub-{subject}_ses-{session}_task-mapper_desc-r2_space-T1w_pars.nii.gz')


def load_r2_img(subject, session, bids_folder=BIDS_FOLDER, deriv=DERIV):
    fn = _r2_path(subject, session, bids_folder, deriv)
    return nib.load(str(fn)) if fn.exists() else None


def load_r2(subject, session, bids_folder=BIDS_FOLDER, deriv=DERIV, mask_img=None):
    """Return a flat R² array for one (subject, session).  If mask_img is given,
    restrict to that ROI (resampled to the R² map grid, nearest)."""
    img = load_r2_img(subject, session, bids_folder, deriv)
    if img is None:
        return None
    data = img.get_fdata().ravel()
    if mask_img is None:
        return data[data != 0]
    m = image.resample_to_img(mask_img, img, interpolation='nearest')
    return data[m.get_fdata().ravel() > 0]

## Whole-brain R² (voxels in the fitting mask, R² ≠ 0)

In [ ]:
subjects = sorted({p.name.split('sub-')[1]
                   for p in (BIDS_FOLDER / 'derivatives' / DERIV).iterdir()
                   if p.is_dir() and p.name.startswith('sub-')})
print('subjects:', subjects)

rows = []
for sub in subjects:
    for ses in SESSIONS:
        r2 = load_r2(sub, ses)
        if r2 is None:
            continue
        rows.append(pd.DataFrame({
            'subject': sub, 'session': ses,
            'field_strength': '7T' if '7t' in ses else '3T',
            'r2': r2,
        }))
df = pd.concat(rows, ignore_index=True)
df.groupby(['subject', 'session']).size()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df[df['r2'] > 0], x='session', y='r2',
               hue='field_strength', ax=ax, order=SES_ORDER,
               inner='quartile', cut=0)
ax.set_yscale('log')
ax.set_ylabel('R² (voxels with R² > 0)')
ax.set_title('Numerosity-PRF R² per scanner session — whole-brain')
plt.tight_layout()

## ROI-restricted R² (NPC / NPCl / NPCr)

Uses the subject-specific T1w NPC masks produced by `balgrist/surface/get_npc_mask.py`. Masks live at:
`derivatives/masks/sub-<s>/anat/sub-<s>_space-T1w_desc-{NPC,NPCl,NPCr}_mask.nii.gz`

In [ ]:
def _mask_path(subject, roi, bids_folder=BIDS_FOLDER):
    return (bids_folder / 'derivatives' / 'masks'
            / f'sub-{subject}' / 'anat'
            / f'sub-{subject}_space-T1w_desc-{roi}_mask.nii.gz')


def load_mask(subject, roi, bids_folder=BIDS_FOLDER):
    fn = _mask_path(subject, roi, bids_folder)
    return nib.load(str(fn)) if fn.exists() else None


roi_rows = []
for sub in subjects:
    for roi in ('NPC', 'NPCl', 'NPCr'):
        mask = load_mask(sub, roi)
        if mask is None:
            continue
        for ses in SESSIONS:
            r2 = load_r2(sub, ses, mask_img=mask)
            if r2 is None or len(r2) == 0:
                continue
            roi_rows.append(pd.DataFrame({
                'subject': sub, 'session': ses,
                'field_strength': '7T' if '7t' in ses else '3T',
                'roi': roi, 'r2': r2,
            }))
roi_df = pd.concat(roi_rows, ignore_index=True)
roi_df.groupby(['subject', 'roi', 'session']).size().unstack('session').fillna(0).astype(int)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, roi in zip(axes, ('NPC', 'NPCl', 'NPCr')):
    sub = roi_df[(roi_df['roi'] == roi) & (roi_df['r2'] > 0)]
    if sub.empty:
        ax.set_title(f'{roi} — no data')
        continue
    sns.violinplot(data=sub, x='session', y='r2', hue='field_strength',
                   ax=ax, order=SES_ORDER, inner='quartile', cut=0)
    ax.set_yscale('log')
    ax.set_title(f'{roi}')
    ax.tick_params(axis='x', rotation=30)
    if ax is not axes[0]:
        ax.set_ylabel('')
axes[0].set_ylabel('R² (voxels with R² > 0, in ROI)')
plt.suptitle('NPC-restricted numerosity-PRF R² per scanner session', y=1.02)
plt.tight_layout()

In [ ]:
# ROI summary table: fraction of NPC voxels with R² > 0.05 per (subject, session, roi)
roi_summary = (roi_df.groupby(['subject', 'session', 'roi'])
                     .agg(n_vox=('r2', 'size'),
                          median_r2=('r2', 'median'),
                          frac_above_05=('r2', lambda s: float((s > 0.05).mean())),
                          frac_above_10=('r2', lambda s: float((s > 0.10).mean())),
                          p95_r2=('r2', lambda s: float(np.quantile(s, 0.95))))
                     .reset_index())

print('=== NPCr: fraction of voxels with R² > 0.05 ===')
print(roi_summary[roi_summary['roi'] == 'NPCr']
      .pivot(index='subject', columns='session', values='frac_above_05')
      .round(3))
print('\n=== NPCr: 95th-percentile R² ===')
print(roi_summary[roi_summary['roi'] == 'NPCr']
      .pivot(index='subject', columns='session', values='p95_r2')
      .round(3))
print('\n=== NPCl: fraction of voxels with R² > 0.05 ===')
print(roi_summary[roi_summary['roi'] == 'NPCl']
      .pivot(index='subject', columns='session', values='frac_above_05')
      .round(3))

## Whole-brain tail mass across thresholds

In [ ]:
thresholds = [0.01, 0.02, 0.05, 0.10, 0.20]
tail_rows = []
for (sub, ses), g in df.groupby(['subject', 'session']):
    for t in thresholds:
        tail_rows.append({'subject': sub, 'session': ses, 'threshold': t,
                          'frac': float((g['r2'] > t).mean())})
tail = pd.DataFrame(tail_rows)

fig, axes = plt.subplots(1, len(thresholds), figsize=(4 * len(thresholds), 4),
                         sharey=True)
for ax, t in zip(axes, thresholds):
    sns.barplot(data=tail[tail['threshold'] == t], x='session', y='frac', ax=ax,
                order=SES_ORDER, errorbar='sd')
    ax.set_title(f'R² > {t}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('fraction of voxels')
plt.tight_layout()

## Paired within-subject Balgrist 3T vs 7T (NPCr)

If the NPC effect survives the 3T-vs-7T contrast specifically in numerosity parietal cortex, this is stronger evidence one way or the other.

In [ ]:
paired = (roi_summary[(roi_summary['roi'] == 'NPCr') &
                      (roi_summary['session'].isin(['balgrist3t', 'balgrist7t']))]
          .pivot(index='subject', columns='session',
                 values=['frac_above_05', 'median_r2', 'p95_r2']))
paired

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in [(axes[0], 'median_r2', 'NPCr median R²'),
                        (axes[1], 'frac_above_05', 'NPCr fraction R² > 0.05'),
                        (axes[2], 'p95_r2', 'NPCr 95th-percentile R²')]:
    wide = (roi_summary[(roi_summary['roi'] == 'NPCr') &
                        (roi_summary['session'].isin(['balgrist3t', 'balgrist7t']))]
             .pivot(index='subject', columns='session', values=col))
    keep = ['balgrist3t', 'balgrist7t']
    wide = wide[keep].dropna()
    for sub_id, row in wide.iterrows():
        ax.plot(keep, row.values, marker='o', label=f'sub-{sub_id}')
    ax.set_title(title)
    ax.set_xlabel('')
axes[0].legend(fontsize=8, loc='best')
plt.tight_layout()